# `logging` — remplacer les `print` par de vrais journaux

En développement, on met des `print` partout. En production, c'est ingérable :
impossible de les désactiver, de les filtrer, de les dater ou de les envoyer dans un fichier
sans modifier le code.

Le module `logging` (fourni avec Python) résout tout ça. Un **log** (journal), c'est un message
horodaté avec un **niveau de gravité**, qu'on peut router où on veut (console, fichier, les deux...)
et filtrer **sans toucher au code**.

| `print` | `logging` |
|---|---|
| toujours affiché | filtrable par niveau, sans changer le code |
| pas de date, pas d'origine | horodatage + nom du module automatiques |
| console uniquement | console, fichier, réseau... et plusieurs à la fois |
| à effacer avant la mise en prod | reste en place, simplement réglé plus ou moins verbeux |

---
## 1. Les 5 niveaux de gravité

Chaque message a un niveau. On règle ensuite un **seuil** : tout ce qui est en dessous est ignoré.

| Niveau | Quand l'utiliser | Exemple |
|---|---|---|
| `DEBUG` | détail technique pour le développeur | « requête SQL : SELECT... » |
| `INFO` | vie normale de l'application | « utilisateur connecté » |
| `WARNING` | anormal mais l'app continue | « disque presque plein » |
| `ERROR` | une opération a échoué | « échec d'envoi de l'email » |
| `CRITICAL` | l'application ne peut plus fonctionner | « base de données injoignable » |

> Par défaut, le seuil est `WARNING` : les `DEBUG` et `INFO` sont ignorés.
> C'est pour ça qu'un `logging.info(...)` « ne s'affiche pas » chez les débutants.

In [ ]:
import logging

# basicConfig : configuration en une ligne (à faire UNE fois, au démarrage du programme)
# force=True : utile en notebook pour pouvoir re-exécuter la cellule (remplace la config précédente)
logging.basicConfig(
    level=logging.DEBUG,                                  # seuil : tout afficher
    format="%(asctime)s [%(levelname)-8s] %(message)s",   # date [NIVEAU] message
    datefmt="%H:%M:%S",
    force=True,
)

logging.debug("Détail technique : cache initialisé avec 128 entrées")
logging.info("L'application démarre")
logging.warning("Le fichier de config est absent, valeurs par défaut utilisées")
logging.error("Impossible d'envoyer l'email de confirmation")
logging.critical("Base de données injoignable — arrêt")

In [ ]:
import logging

# Même code, seuil remonté à WARNING : les messages DEBUG et INFO disparaissent
# → c'est ça la force du logging : on filtre SANS modifier les lignes de log
logging.basicConfig(
    level=logging.WARNING,
    format="%(asctime)s [%(levelname)-8s] %(message)s",
    datefmt="%H:%M:%S",
    force=True,
)

logging.debug("Invisible maintenant")
logging.info("Invisible aussi")
logging.warning("Toujours visible")
logging.error("Toujours visible")

---
## 2. La bonne pratique : un logger par module

Dans un vrai projet, on ne loggue pas via `logging.info(...)` directement : chaque fichier
crée **son** logger avec `logging.getLogger(__name__)`. Deux avantages :

1. Chaque message indique **de quel module** il vient (`__name__` = le nom du fichier/module).
2. On peut régler le niveau **module par module** (ex : rendre bavarde seulement la partie paiement).

C'est LA ligne qu'on retrouve en tête de quasiment tous les fichiers Python professionnels :
```python
logger = logging.getLogger(__name__)
```

In [ ]:
import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)-8s] %(name)s — %(message)s",   # %(name)s = origine du message
    datefmt="%H:%M:%S",
    force=True,
)

# Simulation de deux « modules » d'une même application
logger_paiement = logging.getLogger("boutique.paiement")
logger_stock = logging.getLogger("boutique.stock")

logger_paiement.info("Paiement de 49.99 € accepté")
logger_stock.warning("Stock du produit 'Clavier' sous le seuil : 3 restants")

# Régler UN module sans toucher aux autres :
logger_stock.setLevel(logging.ERROR)      # le module stock ne parlera plus qu'en cas d'erreur
logger_stock.warning("Ce warning est désormais filtré")   # invisible
logger_paiement.info("Le module paiement, lui, parle toujours")

---
## 3. Journaliser les exceptions — `logger.exception`

Dans un `except`, utiliser `logger.exception(...)` : c'est un `error(...)` qui ajoute
**automatiquement la traceback** (la trace complète de l'erreur : fichier, ligne, enchaînement
des appels). Indispensable pour diagnostiquer après coup un incident en production.

In [ ]:
import logging

logging.basicConfig(
    level=logging.INFO,
    format="[%(levelname)s] %(message)s",
    force=True,
)
logger = logging.getLogger("demo")

commande = {"produit": "Clavier", "quantite": 3}

try:
    prix_total = commande["prix"] * commande["quantite"]   # 'prix' n'existe pas → KeyError
except KeyError:
    logger.exception("Commande invalide, champ manquant")
    # ↑ le message + la traceback complète, sans rien faire de plus

print("\nL'application continue de tourner — l'incident est tracé, pas ignoré.")

---
## 4. Écrire les logs dans un fichier (et la console en même temps)

En production, les logs vont dans un **fichier** qu'on peut consulter après coup.
Les **handlers** (aiguillages) permettent d'envoyer chaque message vers plusieurs destinations —
ici : tout dans le fichier, seulement les problèmes à l'écran.

In [ ]:
import logging
from pathlib import Path

fichier_log = Path("application.log")

# Deux destinations (handlers) :
vers_fichier = logging.FileHandler(fichier_log, encoding="utf-8")
vers_fichier.setLevel(logging.DEBUG)                     # le fichier reçoit TOUT

vers_console = logging.StreamHandler()
vers_console.setLevel(logging.WARNING)                   # la console ne montre que les problèmes

logging.basicConfig(
    level=logging.DEBUG,
    format="%(asctime)s [%(levelname)-8s] %(message)s",
    handlers=[vers_fichier, vers_console],
    force=True,
)

logging.info("Démarrage du traitement des commandes")     # fichier seulement
logging.debug("42 commandes chargées")                    # fichier seulement
logging.error("Commande n°17 rejetée : adresse invalide") # fichier + console

# Vérifions le contenu du fichier :
print("\n─── contenu de application.log ───")
print(fichier_log.read_text(encoding="utf-8"))

In [ ]:
# ── Nettoyage : fermer les handlers et supprimer le fichier de démo ──
import logging
from pathlib import Path

for handler in logging.getLogger().handlers[:]:
    handler.close()
    logging.getLogger().removeHandler(handler)

Path("application.log").unlink(missing_ok=True)
print("Nettoyage fait.")

---
## Récapitulatif

- 5 niveaux : `DEBUG` < `INFO` < `WARNING` (seuil par défaut) < `ERROR` < `CRITICAL`.
- `logging.basicConfig(level=..., format=...)` : une fois, au démarrage (+ `force=True` en notebook).
- Dans chaque module : `logger = logging.getLogger(__name__)`.
- Dans un `except` : `logger.exception(...)` = message + traceback automatique.
- Les handlers routent les messages : fichier exhaustif + console réservée aux problèmes.
- On règle la verbosité **sans modifier le code** — c'est tout l'intérêt face à `print`.

> **En entretien** : « `print` est pour le développement, `logging` pour la vie de l'application :
> niveaux filtrables, horodatage, origine du message, destinations multiples. Un logger par module
> via `getLogger(__name__)`, et `logger.exception` dans les `except` pour garder la traceback. »